In [3]:
from calc_setup import *
set_design_code('ec2_2004')


In [4]:
%%render
# Parameters

# --- 3. GET CODE PROPERTIES (structuralcodes) ---
# We create a concrete material to get the design strength (fcd) automatically # structuralcodes handles the partial factors (gamma_c) for us.
b = 400 * mm
h = 400 * mm
f_ck = 30 * MPa
f_yk = 500 * MPa
c_nom = 30 * mm
bar_dia = 20 * mm
n_bars = 4
conc=create_concrete(fck=f_ck.value) 

<IPython.core.display.Latex object>

In [5]:
%%render

f_cd = conc.fcd() * MPa  # Convert back to unit-aware variable

A_c = b * h
A_s = n_bars * (pi * (bar_dia/2)**2)
N_Rd = 0.85 * f_cd * (A_c - A_s) + f_yk * A_s

<IPython.core.display.Latex object>

## Singly reinforced RC beam design to EC2

In [25]:
%%render params
# EC2 ULS bending resistance for a singly reinforced rectangular section
# Units: MPa, mm, mm^2  -> result in kN·m

# Inputs
f_ck = 30    # concrete characteristic strength
f_yk = 460.0     # steel characteristic yield strength
gamma_c = 1.5
gamma_s = 1.15
alpha_cc = 0.85

b = 280.0        # section width
d = 380        # effective depth
A_s = 1470     # tensile steel area
E_s = 200000.0   # MPa

# Design strengths
f_cd = alpha_cc * f_ck / gamma_c
f_yd = f_yk / gamma_s

# EC2 equivalent rectangular stress block (EN 1992-1-1, fck <= 90 MPa)
eta = 1.0 if f_ck <= 50 else 1.0 - (f_ck - 50.0) / 200.0
lam = 0.8 if f_ck <= 50 else 0.8 - (f_ck - 50.0) / 400.0
# Neutral axis from force equilibrium: As*fyd = eta*fcd*b*lam*x
x = A_s * f_yd / (eta * f_cd * b * lam)

# Lever arm and moment resistance
z = d - 0.5 * lam * x
z_max = 0.95 * d
z_use = min(z, z_max)

M_Rd_Nmm = A_s * f_yd * z_use



<IPython.core.display.Latex object>

In [28]:
%%render params
M_Rd_kNm = M_Rd_Nmm / 1e6 #KNm



<IPython.core.display.Latex object>

In [21]:
%%render
# Optional ductility check (steel yielding)
eps_cu3 = 3.5e-3 if f_ck <= 50 else (2.6 + 35.0 * ((90.0 - f_ck) / 100.0) ** 4) / 1000.0
eps_yd = f_yd / E_s
xi = x / d
xi_lim = eps_cu3 / (eps_cu3 + eps_yd)



<IPython.core.display.Latex object>

In [11]:
# Cell 3
print(f"M_Rd = {M_Rd_kNm:.2f} kN·m")
print(f"x/d = {xi:.3f},  xi_lim = {xi_lim:.3f}")
if xi > xi_lim:
    print("Warning: section may be over-reinforced (steel may not yield).")



M_Rd = 284.29 kN·m
x/d = 0.361,  xi_lim = 0.617


In [ ]:
%%render

# **General calculation**
### Calculation Report  
**Author:** Vance Kang
**Date:** {{}}  
**Project No.:** 

---

## Design Loading calculations


### Simply supported beam with UDL


In [ ]:
%%render params
L = 13        # Span length, $m$
w = 0.209      # UDL, $kN/m$
E = 200e6      # Young's modulus, $kN/m^2$ (200 GPa = $200e6 \; kN/m^2$)
I = 7.05 * 10**-6     # Second moment of area, $m^4$
x =  L/2        # Position along beam, $m$

# ----------------------------------------------------


<IPython.core.display.Latex object>

![My Image](img/simplebeam.png)

In [ ]:
%%render
R = w * L / 2
M_max = w*L**2 / 8
delta_max = 5*w*L**4 / (384*E*I)

<IPython.core.display.Latex object>

### simply supported with 1 concentrated loads


In [ ]:
%%render params
L = 10        # span length, ℓ (m)
P = 12       # each point load (kN)
P_ult = P * 1.5 *1.2 # factored load (kN)
dd

# -------------------------------------------------------

<IPython.core.display.Latex object>

In [15]:
%%render
R = P_ult / 2
M_max = P_ult * L / 4

<IPython.core.display.Latex object>

# -------------------------------------------------------
### simply supported with 2 concentrated loads
# -------------------------------------------------------

In [7]:
%%render params
L = 6        # span length, ℓ (m)
P = 6       # each point load (kN)
a = 1        # distance of each load from the support (m)

E = 200e6      # Young's modulus (kN/m²)  200 GPa = 200e6 kN/m²
I = 8.5e-5     # second moment of area (m⁴)

x_left = 2.5   # a point with x < a   (m)
x_mid  = L/2   # a point with a < x < (L - a)   (m)
# -------------------------------------------------------



<IPython.core.display.Latex object>

![My Image](img/simplebeam2concentratedloads.png)

In [8]:
%%render
R = P # R = V = P
M_max = P * a # Maximum moment between loads 
delta_max = (P * a)/(24 * E * I) * (3*L**2 - 4*a**2)  # Maximum deflection at center


<IPython.core.display.Latex object>

### Grating Loading Calculation

In [ ]:

%%render
rho = 80  #  steel grating area load assumed (kg/m2)
w_DL = rho * 10*10**-3
LL = 2.5    #assumed live load
LW = 1 #load width(m)

#per m run loading
q = (1.35*w_DL + 1.5*LL) * LW  # kN/m

load__per__anchor = q/(1000/300) #(KN) spaced every 300mm

<IPython.core.display.Latex object>

### Portal frame Moment

![My Image](img/PortalFrameMoment.png)

In [ ]:
%%render params
# Example usage (edit values)
w = 10.0 #projected loading kN/m
L = 10.0 #Total span m 
I_R = 1.0 #reference moment of inertia
I_C = 1.5 * I_R   # preliminary assumption from the note
h = 3.0 #height up to lowest rafter m
f = 2 #rise of rafter m

<IPython.core.display.Latex object>

In [ ]:
%%render
s = sqrt((L/2)**2 + f**2) #length of rafter m
k   = (I_R * h) / (I_C * s)
phi = f / h
m   = 1 + phi
B   = 2 * (k + 1) + m
C   = 1 + 2 * m
N   = B + m * C

<IPython.core.display.Latex object>

In [ ]:
%%render
M_E = w * L**2 * (3 + 5*m) / (16 * N)
M_A = w * L**2 / 8 + m * M_E


<IPython.core.display.Latex object>

In [ ]:
%%render
M_A = w * L**2 / 8

<IPython.core.display.Latex object>